In [ ]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True,
               cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import scipy as sp
import pandas as pd

import matplotlib.pyplot as plt

%matplotlib inline

sys.path.extend([r'/mnt/storage_ssd_raid/research_projects/snaplab_tools'])

from src import nulls
from src.notebook import load_notebook_data
from src.plotting import figure_scratch_dir, setup_notebook_style
from snaplab_tools.plotting.utils import save_panel
from src.timescales import (convert_data_dict, average_data_dict_converted,
                             compute_regional_corrs, compute_deflections,
                             create_ints_dataframe_custom_grouping)
from src.gams import apply_region_mask, build_nuisance
from snaplab_tools.plotting.plotting import plot_correlation
from snaplab_tools.stats import partial_pearsonr

setup_notebook_style()


## Panel + analysis support

Helpers for this notebook, gathered so the cells below stay readable. Collapse it.

In [ ]:
# ── Panel + analysis support ────────────────────────────────────────────────────────────────
# Every helper this notebook defines, collected in one place so the analysis below reads as a
# sequence of steps rather than a mix of steps and machinery. Collapse this cell when reading.
# These are notebook-specific by design: anything used by a second notebook belongs in src/.
# They resolve globals (config constants, loaded data) at CALL time, so definitions may refer
# to names bound further down the notebook.

def defl_stats(rest, task):
    """(mean, variability) across subjects for one rest/task pair, honouring the toggles.

    Variability is std(d) / mean|d| under 'cv' and std(d) under 'std'. The CV denominator
    is always the magnitude mean so the statistic stays finite and comparable whether or
    not DEFLECTION_ABS is set (see the note above).
    """
    d = compute_deflections(rest, task, absolute=DEFLECTION_ABS)
    mean_d = np.nanmean(d, axis=0)
    std_d  = np.nanstd(d, axis=0)
    if VARIABILITY_STAT == 'std':
        return mean_d, std_d
    return mean_d, std_d / np.nanmean(np.abs(compute_deflections(rest, task)), axis=0)


def _stack_mean(d, patterns):
    arrays = [d[k] for k in d if any(p in k for p in patterns) and d[k] is not None]
    return np.nanmean(np.stack(arrays, axis=0), axis=0) if arrays else None


def _to_full(vec_masked):
    """Re-expand a masked (valid-region) vector to the full 400-parcel grid (NaN at masked regions)."""
    full = np.full(n_regions_full, np.nan)
    full[~region_mask] = vec_masked
    return full


def _make_surrogates(vec_full, name):
    return nulls.generate_surrogates(
        vec_full, name=name, n_regions=n_regions_full, n_perms=N_PERMS, kind='geodesic',
        distance_matrix=distance_matrix, hemi=hemi_labels,
        cache_dir=surrogate_cache_dir)


def _panel_null(y_full, panel_slug):
    """Spatial-null test of corr(y-axis map, S-A axis) for one panel, honouring NULL_TARGET.

    Returns corr_with_null's dict (r, p_smash, n, null_lo, null_hi, null). The observed r is
    identical under both targets -- correlation is symmetric -- so only p_smash moves.
    """
    if NULL_TARGET == 'sa':
        return nulls.corr_with_null(y_full, sa_axis_full, surr_sa, method=CORR_METHOD)
    surr_y = _make_surrogates(y_full, f'{_YMAP_KEY}-{panel_slug}')
    return nulls.corr_with_null(sa_axis_full, y_full, surr_y, method=CORR_METHOD)


def _row_subjects(processor, dd, tasks):
    """Subject order of convert_data_dict's stacked rows, verified consistent across `tasks`.

    convert_data_dict stacks one row per scan in registry order and silently skips scans that
    are absent from `dd`, so the row -> subject mapping is only well defined when every
    constituent scan of an averaged group yields the same subject sequence. Checked, not assumed.
    """
    orders = [[s.subject_id for s in processor.scan_registry
               if s.task == t and str(s) in dd] for t in tasks]
    for o in orders[1:]:
        if o != orders[0]:
            raise ValueError('constituent scans disagree on subject order')
    return np.asarray(orders[0], dtype=str)


def _reindex(subs, wanted):
    pos = {s: i for i, s in enumerate(subs)}
    return np.array([pos[s] for s in wanted])


def _half_violin(ax, vals, pos, side, color, width=0.75, floor=None):
    """One half of a split violin at `pos`, with a median rule on the same side.

    `floor` clips the kernel density at a hard bound. |r| cannot be negative, so without it
    the KDE puts visible mass below zero and the violin reads as if the data went there.
    """
    vp = ax.violinplot(vals, positions=[pos], widths=width,
                       showextrema=False, showmedians=False)
    for body in vp['bodies']:
        verts = body.get_paths()[0].vertices
        verts[:, 0] = (np.clip(verts[:, 0], -np.inf, pos) if side == 'left'
                       else np.clip(verts[:, 0], pos, np.inf))
        if floor is not None:
            verts[:, 1] = np.clip(verts[:, 1], floor, None)
        body.set_facecolor(color)
        body.set_edgecolor('none')
        body.set_alpha(0.85)
    med = np.nanmedian(vals)
    x0, x1 = (pos - width / 2, pos) if side == 'left' else (pos, pos + width / 2)
    ax.plot([x0, x1], [med, med], color='0.15', lw=1.0, solid_capstyle='butt', zorder=3)
    return med


In [ ]:
dataset = 'HCP-YA'
which_modality = 'FMRI'
use_7t = False
output_type = 'timescale_auc_zero_crossing'
# output_type = output_type + '_bp-0.01-0.08hz'

use_resids = True
notebook_data = load_notebook_data(dataset=dataset, which_modality=which_modality,
                                   use_7t=use_7t, output_type=output_type, use_resids=use_resids,
                                   load_subject_data=False)

In [ ]:
nas_scratch_dir = figure_scratch_dir(dataset)

In [ ]:
# get key outputs for this notebook
timescales_processor = notebook_data['timescales_processor']
data_dict = notebook_data['data_dict']
data_dict_tsnr = notebook_data['data_dict_tsnr']
results_dir = notebook_data['results_dir']

df_neuro = notebook_data['supporting_data']['df_neuro']
yeo_systems = notebook_data['supporting_data']['yeo_systems']

print(data_dict.keys())
print(df_neuro.columns)

In [ ]:
n_regions = len(yeo_systems)
n_systems = len(np.unique(yeo_systems))

In [ ]:
subject_ids = list(dict.fromkeys(scan.subject_id for scan in timescales_processor.scan_registry))
n_subs = len(subject_ids)
print(f'Number of subjects: {n_subs}')

In [ ]:
timescales_processor.scan_registry[:5]

In [ ]:
if dataset == 'HCP-YA':
    scans_to_keep = ['rfMRIREST1LR', 'rfMRIREST1RL', 'rfMRIREST2LR', 'rfMRIREST2RL']
    # scans_to_keep = ['rfMRIREST1LR', 'rfMRIREST1RL']
    # scans_to_keep = ['rfMRIREST1LR', 'rfMRIREST2LR']
    condition_name = 'WM' # 'ALL', 'WM', 'EMOTION', 'GAMBLING', 'LANGUAGE', 'RELATIONAL', 'SOCIAL'

    if 'ALL' in condition_name:
        tasks = [
                'tfMRIWMLR', 'tfMRIWMRL',
                'tfMRIEMOTIONLR', 'tfMRIEMOTIONRL',
                'tfMRIGAMBLINGLR', 'tfMRIGAMBLINGRL',
                'tfMRILANGUAGELR', 'tfMRILANGUAGERL',
                'tfMRIRELATIONALLR', 'tfMRIRELATIONALRL',
                'tfMRISOCIALLR', 'tfMRISOCIALRL',
                ]
        scans_to_keep = scans_to_keep + tasks
        task_performance = []
    elif 'WM' in condition_name:
        scans_to_keep.append('tfMRIWMLR')
        scans_to_keep.append('tfMRIWMRL')
        task_performance = ['WM_Task_Acc', 'WM_Task_Median_RT']
    elif 'EMOTION' in condition_name:
        scans_to_keep.append('tfMRIEMOTIONLR')
        scans_to_keep.append('tfMRIEMOTIONRL')
        task_performance = ['EMOTION_Task_Acc', 'EMOTION_Task_Median_RT']
    elif 'GAMBLING' in condition_name:
        scans_to_keep.append('tfMRIGAMBLINGLR')
        scans_to_keep.append('tfMRIGAMBLINGRL')
        task_performance = ['GAMBLING_Task_Perc_Larger', 'GAMBLING_Task_Perc_Smaller']
    elif 'LANGUAGE' in condition_name:
        scans_to_keep.append('tfMRILANGUAGELR')
        scans_to_keep.append('tfMRILANGUAGERL')
        task_performance = ['LANGUAGE_Task_Acc', 'LANGUAGE_Task_Median_RT']
    elif 'RELATIONAL' in condition_name:
        scans_to_keep.append('tfMRIRELATIONALLR')
        scans_to_keep.append('tfMRIRELATIONALRL')
        task_performance = ['RELATIONAL_Task_Acc', 'RELATIONAL_Task_Median_RT']
    elif 'SOCIAL' in condition_name:
        scans_to_keep.append('tfMRISOCIALLR')
        scans_to_keep.append('tfMRISOCIALRL')
        task_performance = ['SOCIAL_Task_Perc_Random', 'SOCIAL_Task_Perc_TOM'] 

    # scan_registry_filtered = pd.DataFrame(columns=timescales_processor.scan_registry.columns)
    # for scan in scans_to_keep:
    #     # print(scan)
    #     subject_filter = timescales_processor.scan_registry['task'] == scan
    #     scan_registry_filtered = pd.concat((scan_registry_filtered, timescales_processor.scan_registry.loc[subject_filter])).reset_index(drop=True)
        
    # timescales_processor.scan_registry = scan_registry_filtered

In [ ]:
timescales_processor.match_registry_to_behavior()
print(timescales_processor.df_behavior_matched.shape)

In [ ]:
df_behavior = timescales_processor.df_behavior_matched.copy()
df_behavior.head()

# Data for figures

In [ ]:
brain_map = 'SA_Axis'

data_dict_converted = convert_data_dict(data_dict, timescales_processor.scan_registry)
data_dict_converted = average_data_dict_converted(data_dict_converted, average_3t_rest=True, average_3t_task_pairs=True)
data_dict_converted.keys()

## Subject-level analyses: regional

## INT deflection variance and covariance

In [ ]:
# ── Deflection definition and rest-INT control ────────────────────────────────
# INT deflection = task - rest, per subject per region. No rest control is baked into the
# deflection itself (compute_deflections has no such option): regressing rest out of
# (task - rest) is algebraically identical to residualising task on rest, which turns the
# deflection into something else entirely. Rest INT is controlled at the analysis step,
# below, and only for the behavioural correlations.
#
# DEFLECTION_ABS   False -> signed (task - rest). DEFAULT. Task shortens INT in ~95% of
#                           subject x region cells (see diagnostics), so the deflection is
#                           near-unidirectional and the signed value reads directly as
#                           "amount of INT shortening" (negative = task shortens INT).
#                  True  -> |task - rest|, magnitude only. Flips the sign of essentially
#                           every correlation below, since |d| ~ -d here.
#
# VARIABILITY_STAT 'cv'  -> top-row panel 2 shows CV = std(d) / mean|d| across subjects.
#                           DEFAULT: scale-free, so it is not mechanically inflated in
#                           long-INT regions the way raw std is.
#                  'std' -> raw std across subjects. NOTE the diagnostics show raw std
#                           carries NO S-A structure beyond the mean panel
#                           (partial r ~ -0.00, p ~ 0.99) -- it is near-collinear with the
#                           mean (r ~ +0.90), so the two panels say one thing twice.
#
# The CV denominator is mean|d| (always positive and bounded away from zero) rather than
# mean(d). Under DEFLECTION_ABS=True that is the textbook CV. Under signed deflections a
# mean(d) denominator would be unusable: mean signed deflection reaches -0.0025 in one
# parcel and is positive in 3 of 360, so std/mean explodes (max ~99) and flips sign.
DEFLECTION_ABS   = True   # False (signed) | True (absolute)
VARIABILITY_STAT = 'std'    # 'cv' | 'std'
# ─────────────────────────────────────────────────────────────────────────────

if VARIABILITY_STAT not in ('cv', 'std'):
    raise ValueError(f"VARIABILITY_STAT must be 'cv' or 'std'; got {VARIABILITY_STAT!r}")

# Neural data for rest and task
rest_data = data_dict_converted['task-REST_3T']           # (n_subjects, n_regions)
task_data = data_dict_converted[f'task-{condition_name}'] # (n_subjects, n_regions)

behavior_measures = [
    f'{condition_name}_Task_Acc',
    f'{condition_name}_Task_Median_RT',
    'CogTotalComp_Unadj',
]

# Build tSNR mask: exclude lowest 10% tSNR regions for the task condition
df_tsnr = create_ints_dataframe_custom_grouping(data_dict_tsnr, average_3t_task_pairs=True)
tsnr_col = f'task-{condition_name}'
region_mask = (df_tsnr[tsnr_col] < np.quantile(df_tsnr[tsnr_col], q=0.1)).values

# Apply mask — drops low-tSNR regions from data arrays and SA axis
sa_axis_masked = df_neuro[brain_map].values.copy()
rest_data, task_data, sa_axis_masked = apply_region_mask(
    region_mask, rest_data, task_data, sa_axis=sa_axis_masked)
yeo_systems_masked = np.asarray(yeo_systems)[~region_mask]


deflection_matrix = compute_deflections(rest_data, task_data, absolute=DEFLECTION_ABS)
mean_deflection, var_deflection = defl_stats(rest_data, task_data)

# Rest-INT control for the behavioural correlations (bottom row): each subject's MEAN rest
# INT across regions. One scalar per subject, constant across regions, so it removes global
# INT differences without reshaping the regional map -- the S-A coupling survives. Matches
# the nuisance covariate used by fig3's GAMs. Per-region rest control is deliberately NOT
# used; the diagnostics cell shows it collapses the S-A coupling (r +0.56 -> -0.18).
subj_mean_rest = np.nanmean(rest_data, axis=1)   # (n_subjects,)

# The covariate matrix passed to compute_regional_corrs: age, sex, head motion, and subject
# mean rest INT. build_nuisance is the shared owner of the nuisance definition (fig3/fig4
# call it too), so fig2 cannot drift from it. It appends the row-mean of `rest_data` itself, so
# the last column IS subj_mean_rest -- note rest_data is already tSNR-masked here, which is
# fig2's convention and differs from the full-parcellation matrix fig3/fig4 must pass (see
# build_nuisance's docstring).
#
# Age and sex are genuine confounds of behaviour here (|r| up to 0.16 with WM accuracy). Motion
# is the mean relative RMS displacement HCP reports per run -- NOT Power's FD; report it as
# such. Rest and task motion enter as separate columns rather than one average: they correlate
# at r ~ 0.82, but the deflection is a task-minus-rest difference, so a single averaged motion
# term would average over the very contrast the deflection is built on.
_motion_task = ('meanRelRMS_task' if condition_name == 'ALL'   # 'ALL' has no single task column
                else f'meanRelRMS_{condition_name}')
beh_covariates = build_nuisance(df_behavior, rest_data, dataset, include_age=True,
                                fd_columns=['meanRelRMS_rest', _motion_task])
_COV_LAB = f'age, sex, motion (rest + {condition_name}), mean rest INT'

# Axis-label fragments driven by the toggles.
_D_LAB = '|INT Deflection|' if DEFLECTION_ABS else 'INT Deflection'
_V_LAB = 'CV' if VARIABILITY_STAT == 'cv' else 'Std. Dev.'

print(f'deflection         : {"absolute" if DEFLECTION_ABS else "signed"}')
print(f'variability stat   : {VARIABILITY_STAT}')
print(f'deflection_matrix  : {deflection_matrix.shape}')
print(f'mean_deflection    : {mean_deflection.shape}')
print(f'subj_mean_rest     : {subj_mean_rest.shape}  (rest-INT covariate for the bottom row)')
print(f'covariates         : {beh_covariates.shape}  [{_COV_LAB}]')
print(f'                     complete rows: {(~np.isnan(beh_covariates).any(axis=1)).sum()}'
      f'/{beh_covariates.shape[0]}  (NaN rows are dropped per region by partial_pearsonr)')

In [ ]:
# ── Split-half reliability data ───────────────────────────────────────────────
# Phase-encoding split: does the top-row variability statistic replicate across the LR and
# RL acquisitions? Uses defl_stats, so it tracks DEFLECTION_ABS / VARIABILITY_STAT.
_data_raw = convert_data_dict(data_dict, timescales_processor.scan_registry, verbose=False)

_rest_lr = _stack_mean(_data_raw, ['rfMRIREST1LR', 'rfMRIREST2LR'])
_rest_rl = _stack_mean(_data_raw, ['rfMRIREST1RL', 'rfMRIREST2RL'])
_task_lr = _stack_mean(_data_raw, [f'tfMRI{condition_name}LR'])
_task_rl = _stack_mean(_data_raw, [f'tfMRI{condition_name}RL'])

_rest_lr_m = _rest_lr[:, ~region_mask]
_rest_rl_m = _rest_rl[:, ~region_mask]
_task_lr_m = _task_lr[:, ~region_mask]
_task_rl_m = _task_rl[:, ~region_mask]

_, var_defl_lr = defl_stats(_rest_lr_m, _task_lr_m)
_, var_defl_rl = defl_stats(_rest_rl_m, _task_rl_m)

print(f'Split-half r ({_V_LAB}): {float(sp.stats.pearsonr(var_defl_lr, var_defl_rl)[0]):.3f}')

In [ ]:
# ── Spatial nulls for the S-A-axis correlations ───────────────────────────────
# Each panel tests corr(y-axis map, S-A axis) against a spatial-autocorrelation-preserving null:
# per-hemisphere geodesic BrainSMASH variogram surrogates (Burt 2020), cached under
# results/<dataset>/brainsmash_nulls (src.nulls; shared with fig3/fig4).
#
# NULL_TARGET — WHICH map gets surrogated. Correlation is symmetric so the observed r is identical
# either way; only the null distribution (and hence p) changes:
#   'sa'   — surrogate the S-A axis; each panel's INT-derived y-axis map is held fixed. ONE
#            surrogate set is reused by every panel. The paper's design, and the cheap option.
#   'ymap' — surrogate each panel's OWN y-axis map; the S-A axis is held fixed. Every panel gets
#            its own surrogate set, so the null carries that map's spatial autocorrelation rather
#            than the S-A axis's.
#
# The two ask different questions -- 'sa': "could a map with the S-A axis's spatial structure
# correlate this well with this INT map?"; 'ymap': the converse -- and they agree only insofar as
# the two maps have similar autocorrelation. Neither is more correct in general; 'sa' is the
# conventional choice when one map is a fixed reference axis.
#
# Cost of 'ymap' on first run: ~4 min/panel (5 panels, then cached on disk).
#
# Outputs are suffixed whenever this is not the primary (sa) configuration, so a robustness run
# never overwrites the committed figure/CSV.
NULL_TARGET    = 'sa'           # 'sa' | 'ymap'
CORR_METHOD    = 'pearson'

N_PERMS        = 10000
n_regions_full = len(region_mask)   # 400 (pre-mask); masked maps are re-expanded to this length

if NULL_TARGET not in ('sa', 'ymap'):
    raise ValueError(f"NULL_TARGET must be 'sa' or 'ymap', got {NULL_TARGET!r}")

distance_matrix, hemi_labels = nulls.load_distance_matrix(n_regions_full, kind='geodesic')
surrogate_cache_dir = nulls.surrogate_cache_dir(timescales_processor, dataset)

sa_axis_full = df_neuro[brain_map].values   # dense S-A axis over all 400 parcels

# S-A surrogates: only needed under NULL_TARGET='sa' (one set, reused by every panel).
surr_sa = _make_surrogates(sa_axis_full, 'SA_Axis') if NULL_TARGET == 'sa' else None

# Cache key prefix for the per-panel y-axis maps. It must encode everything upstream that changes
# those maps -- condition, signed-vs-absolute, variability statistic -- otherwise switching e.g.
# WM -> EMOTION would silently reuse the previous condition's surrogates from disk under the same
# filename. (dataset is already separated by surrogate_cache_dir.) The -v2 tag retires surrogates
# cached under the old nuisance-regressed deflection definition.
_YMAP_KEY = (f'fig2-{condition_name}-defl{"abs" if DEFLECTION_ABS else "sgn"}'
             f'-{VARIABILITY_STAT}-v2')

print(f'null_target={NULL_TARGET} | n_perms={N_PERMS}')
if NULL_TARGET == 'sa':
    print(f'  S-A surrogates {surr_sa.shape} (n_perms x n_regions), reused by every panel')
else:
    print(f'  per-panel surrogates of each y-axis map; cache key prefix: {_YMAP_KEY}')
    print('  first run generates 5 surrogate sets (~4 min each), cached on disk thereafter')

# Figure 2

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=len(behavior_measures),
                         sharex=False, sharey=False,
                         figsize=(len(behavior_measures)*2.1, 4), dpi=300)

sa_null_rows = []   # per-panel r / p_smash for the S-A-axis correlations -> saved to results_dir

# Every panel tested against a spatial null shows the SPATIAL-null p (custom_pvalue=d['p_smash']),
# never plot_correlation's parametric p: the parametric p ignores spatial autocorrelation and is
# wildly anticonservative on 360 smooth parcels (off by ~10^3 here), and it would disagree with
# the CSV written at the end of this cell. The split-half panel is the one deliberate exception.
#
# _panel_null (defined in the null-config cell) decides WHICH map is surrogated via NULL_TARGET;
# the panel_slug is the on-disk cache key for that panel's y-axis map under NULL_TARGET='ymap'.

# Top row: variance characterization (no behavior). Raw deflections, no rest control — these
# panels describe the deflection itself, so residualising them against rest would answer a
# different question (see the config cell).
variance_panels = [
    (mean_deflection, f'{_D_LAB} (Mean)',   'mean-defl'),
    (var_deflection,  f'{_D_LAB} ({_V_LAB})', f'{VARIABILITY_STAT}-defl'),
]
for i, (stat_data, stat_label, panel_slug) in enumerate(variance_panels):
    d = _panel_null(_to_full(stat_data), panel_slug)
    sa_null_rows.append({'panel': stat_label, 'r': d['r'], 'p_smash': d['p_smash'], 'n': d['n'],
                         'null_target': NULL_TARGET})
    plot_correlation(sa_axis_masked, stat_data, axes[0, i],
                     data_group=yeo_systems_masked, show_stats=True, fontsize=8, method=CORR_METHOD,
                     y_label=stat_label, x_label=' ', size=10, cbar_label='Yeo Systems',
                     show_colorbar=False, data_group_cmap='yeo7',
                     auto_polynomial=False, custom_pvalue=d['p_smash'])

# Top row panel 3: split-half reliability (LR vs RL) — parametric by design, so no
# custom_pvalue: this panel is about measurement reliability, not spatial correspondence, and a
# spatial null does not apply to it.
plot_correlation(var_defl_lr, var_defl_rl, axes[0, 2],
                 data_group=yeo_systems_masked, show_stats=True, fontsize=8, method=CORR_METHOD,
                 y_label=f'{_V_LAB} {_D_LAB} (RL)', x_label=f'{_V_LAB} {_D_LAB} (LR)',
                 size=10, cbar_label='Yeo Systems', show_colorbar=False,
                 data_group_cmap='yeo7', auto_polynomial=False)
# axes[0, 2].set_title('Phase-encoding consistency', fontsize=8, pad=3)

# Bottom row: behavioral covariance. Partial correlations controlling beh_covariates (age, sex,
# head motion, subject mean rest INT; built in the config cell) — every column is subject-level,
# so they remove global confounds without reshaping the regional map the S-A test then reads.
_BEHAVIOR_SLUGS = {'Acc.': 'corr-acc', 'RT': 'corr-rt', 'Cognition': 'corr-cog'}
for i, behavior_measure in enumerate(behavior_measures):
    behavior_data = df_behavior[behavior_measure].values.astype(float)
    rest_correlations, task_correlations, deflection_correlations = compute_regional_corrs(
        rest_data, task_data, behavior_data,
        covariate=beh_covariates, absolute=DEFLECTION_ABS)

    if 'Acc' in behavior_measure:
        y_sublabel = 'Acc.'
    elif 'RT' in behavior_measure:
        y_sublabel = 'RT'
    else:
        y_sublabel = 'Cognition'

    file_label = f'Corr({_D_LAB}, {y_sublabel})'
    d = _panel_null(_to_full(deflection_correlations), _BEHAVIOR_SLUGS[y_sublabel])
    sa_null_rows.append({'panel': file_label, 'r': d['r'], 'p_smash': d['p_smash'], 'n': d['n'],
                         'null_target': NULL_TARGET, 'covariates': _COV_LAB})
    plot_correlation(sa_axis_masked, deflection_correlations, axes[1, i],
                     data_group=yeo_systems_masked, show_stats=True, fontsize=8, method=CORR_METHOD,
                     y_label=file_label, x_label='S-A axis', size=10, cbar_label='Yeo Systems',
                     show_colorbar=False, data_group_cmap='yeo7',
                     auto_polynomial=False, custom_pvalue=max(d['p_smash'], 1 / N_PERMS))

for ax in axes.flat:
    ax.yaxis.set_label_coords(-0.3, 0.5)
    ax.tick_params(axis='y', which='major', pad=1)
    ax.tick_params(axis='x', which='major', pad=1)
    ax.xaxis.labelpad = 2

fig.tight_layout()
# Suffix anything that is not the primary configuration, so a robustness run never overwrites
# the committed figure and CSV. Primary = sa null, signed deflection, CV.
_is_primary  = (NULL_TARGET == 'sa' and not DEFLECTION_ABS and VARIABILITY_STAT == 'cv')
_null_suffix = ''
if not _is_primary:
    _null_suffix = f'_{NULL_TARGET}'
    if DEFLECTION_ABS:
        _null_suffix += '-abs'
    if VARIABILITY_STAT != 'cv':
        _null_suffix += f'-{VARIABILITY_STAT}'
save_panel(fig, nas_scratch_dir, f'corr_ints_sa{_null_suffix}', transparent=True)
plt.show()

# S-A-axis spatial-null statistics (goal 6: results -> results_dir)
pd.DataFrame(sa_null_rows).to_csv(
    os.path.join(results_dir, f'corr_ints_sa_nulls{_null_suffix}.csv'), index=False)
print(pd.DataFrame(sa_null_rows)[['panel', 'r', 'p_smash', 'n']].to_string(index=False))

In [ ]:
# another one just for the colorbar
fig, axes = plt.subplots(nrows=1, ncols=1,
                         sharex=True, sharey=False,
                         figsize=(2, 2), dpi=300)

plot_correlation(sa_axis_masked, mean_deflection, axes, 
                data_group=yeo_systems_masked, show_stats=True, fontsize=8,
                y_label=file_label, x_label='S-A axis', size=10, cbar_label='Yeo Systems', show_colorbar=True, data_group_cmap='yeo7',
                auto_polynomial=True, models_to_test=[1, 2], model_selection_metric='variance_explained')

save_panel(fig, nas_scratch_dir, 'corr_ints_sa_colorbar', transparent=True)
plt.show()

# Supplementary: does task regression change the behavioural correlations?

Figure 2's bottom row is computed from **task-regressed** INTs (`use_resids = True` in the
config cell). This section reloads the same data with `use_resids = False` and puts the two
head to head — one split violin per behaviour, over the distribution of |r| across regions.

Both sides use the same subjects, the same tSNR-masked regions, and the same covariates (age,
sex, head motion, and subject mean rest INT), so the task INTs are the only thing that differs.

Rest INT is a **built-in control**: `regress_task_structure` only ever touches
`condition == 'task'`, so the rest side of the deflection is identical between the two runs.
The compute cell asserts this, and any difference in the panel therefore comes from the task
INTs alone.

In [ ]:
# ── Supp: behavioural correlations with vs without task regression ────────────
# fig2's bottom row is built from task-regressed INTs (use_resids = True in the config cell).
# This reloads the same dataset with use_resids = False and recomputes those correlations, so
# the two pipelines can be put head to head on the same subjects, the same tSNR-masked regions
# and the same rest-INT covariate. Only the task INTs differ between the two sides.
#
# ABS_R  True  -> compare |r| (DEFAULT). The question is whether task regression changes the
#                 STRENGTH of the behavioural coupling, not its direction — and under
#                 DEFLECTION_ABS the sign is close to a global flip anyway (see diagnostics).
#        False -> compare signed r.
ABS_R = True
# ─────────────────────────────────────────────────────────────────────────────

from snaplab_tools.plotting.utils import get_snap_colors

_REST_TASKS = ['rfMRIREST1LR', 'rfMRIREST1RL', 'rfMRIREST2LR', 'rfMRIREST2RL']


# Reload without task regression; every other argument matches the config cell.
notebook_data_noreg = load_notebook_data(
    dataset=dataset, which_modality=which_modality, use_7t=use_7t,
    output_type=output_type, use_resids=False, load_subject_data=False)
proc_noreg      = notebook_data_noreg['timescales_processor']
data_dict_noreg = notebook_data_noreg['data_dict']
proc_noreg.match_registry_to_behavior()

conv_noreg = convert_data_dict(data_dict_noreg, proc_noreg.scan_registry, verbose=False)
conv_noreg = average_data_dict_converted(conv_noreg, average_3t_rest=True,
                                         average_3t_task_pairs=True)
# Same region_mask as the main figure, so both sides describe the same parcels.
rest_noreg, task_noreg, _ = apply_region_mask(
    region_mask, conv_noreg['task-REST_3T'], conv_noreg[f'task-{condition_name}'])

# filter_data_dict drops subjects at the SUBJECT level, and the use_resids=True path filters
# against the residual dict as well, so the two runs can end up with different subject sets
# (a task-GLM failure removes that subject from the regressed run only). Align on the
# intersection before comparing; when the sets already agree this is the identity.
_subs_with    = _row_subjects(timescales_processor, data_dict, _REST_TASKS)
_subs_without = _row_subjects(proc_noreg, data_dict_noreg, _REST_TASKS)
_common = np.asarray([s for s in _subs_with if s in set(_subs_without)])


_iw, _io = _reindex(_subs_with, _common), _reindex(_subs_without, _common)
print(f'subjects: with={len(_subs_with)}  without={len(_subs_without)}  common={len(_common)}')

supp_sides = {
    'With regression':    (rest_data[_iw],  task_data[_iw],  df_behavior.loc[_common]),
    'Without regression': (rest_noreg[_io], task_noreg[_io],
                                proc_noreg.df_behavior_matched.loc[_common]),
}

supp_corrs = {}
for _lab, (_rest, _task, _beh) in supp_sides.items():
    # Same covariate set as the main figure, rebuilt per side so the comparison stays valid
    # even if the two runs ever end up with different subject sets.
    _cov = build_nuisance(_beh, _rest, dataset, include_age=True,
                          fd_columns=['meanRelRMS_rest', _motion_task])
    supp_corrs[_lab] = {
        bm: compute_regional_corrs(_rest, _task, _beh[bm].values.astype(float),
                                   covariate=_cov, absolute=DEFLECTION_ABS)[2]
        for bm in behavior_measures
    }

# Built-in control: regress_task_structure only ever touches condition == 'task', so the rest
# side of the deflection is identical between the two runs and every difference below comes
# from the task INTs alone. A non-zero value here means the two runs are not comparable.
_rest_delta = np.nanmax(np.abs(supp_sides['With regression'][0]
                               - supp_sides['Without regression'][0]))
print(f'rest INT identical across runs: max|diff| = {_rest_delta:.3e}  (expected exactly 0)')

In [ ]:
# ── Supp figure: |r| distributions, with vs without task regression ───────────
# Split violin per behaviour: left half = task-regressed (what fig2's bottom row reports),
# right half = not. Rule = median across regions; Delta = median(with) - median(without).
# Descriptive only — no test is run, because regions are spatially autocorrelated and a
# paired test over 360 parcels would treat them as 360 independent observations.
_SHORT = {behavior_measures[0]: 'Acc.', behavior_measures[1]: 'RT',
          behavior_measures[2]: 'Cognition'}
_snap  = get_snap_colors(normalize=True)
_COL   = {'With regression':    _snap['starry_night_blue'],
          'Without regression': _snap['raspberry_blush']}   # CVD dE 21.5 (comfortable)


fig, ax = plt.subplots(figsize=(3, 2.5), dpi=300)
supp_rows, _vmax = [], 0.0
for i, bm in enumerate(behavior_measures):
    for _side, _lab in (('left', 'With regression'), ('right', 'Without regression')):
        v = supp_corrs[_lab][bm]
        v = np.abs(v) if ABS_R else v
        v = v[~np.isnan(v)]
        _vmax = max(_vmax, v.max())
        _med = _half_violin(ax, v, i, _side, _COL[_lab], floor=0.0 if ABS_R else None)
        supp_rows.append({'behavior': _SHORT[bm], 'condition': _lab,
                          'median': _med, 'mean': v.mean(),
                          'q25': np.percentile(v, 25), 'q75': np.percentile(v, 75),
                          'n_regions': v.size, 'n_subjects': len(_common),
                          'abs_r': ABS_R, 'deflection_abs': DEFLECTION_ABS})

# Headroom so the deltas sit above the widest violin rather than on top of it.
ax.set_ylim(0 if ABS_R else None, _vmax * 1.18)
# for i in range(len(behavior_measures)):
#     _d = supp_rows[2 * i]['median'] - supp_rows[2 * i + 1]['median']
#     ax.annotate(f'$\\Delta$={_d:+.4f}', xy=(i, _vmax * 1.10), ha='center', va='bottom',
#                 fontsize=6, color='0.35')

ax.set_xticks(range(len(behavior_measures)))
ax.set_xticklabels([_SHORT[b] for b in behavior_measures])
ax.set_ylabel(f'{"|r|" if ABS_R else "r"}  ({_D_LAB} vs. behavior)')
ax.tick_params(axis='both', which='major', pad=1)
_handles = [plt.Rectangle((0, 0), 1, 1, fc=_COL[k], alpha=0.85, ec='none') for k in _COL]
ax.legend(_handles, list(_COL), fontsize=6, frameon=False, loc='upper center',
          bbox_to_anchor=(0.5, 1.22), ncol=2, handlelength=1.1, columnspacing=1.2)

fig.tight_layout()
# Suffixed by the toggles so a robustness run never overwrites the committed figure/CSV.
_supp_suffix = ('-abs' if DEFLECTION_ABS else '') + ('' if ABS_R else '-signedr')
save_panel(fig, nas_scratch_dir, f'corr_ints_behavior_taskreg{_supp_suffix}')
plt.show()

pd.DataFrame(supp_rows).to_csv(
    os.path.join(results_dir, f'corr_ints_behavior_taskreg{_supp_suffix}.csv'), index=False)
print(pd.DataFrame(supp_rows)[['behavior', 'condition', 'median', 'mean', 'n_regions']]
      .to_string(index=False))